# Review 2: clean two-task continual learning

Import this notebook into Kaggle, enable **Internet** and a **GPU (T4)**, and attach the private `cic-collection.parquet` using Add Input. Run cells in order. This compares sequential Task 2 fine-tuning with class-balanced replay; it does not poison either arm. The input is a combined flow collection with unverified per-row source provenance. Outputs remain blank until execution.


## 1. Check out the published source revision

The detached SHA contains the existing sampler, model, task builder, replay selector, and forgetting evaluator. `main` does not contain the unified implementation.


In [ ]:
from pathlib import Path
import subprocess, sys
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = 'b2a0afc697272a9d093780057f38c63ee19340b0'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-continual')
if REPO_DIR.exists():
    raise RuntimeError('Repository folder already exists; inspect it or use a fresh Kaggle session.')
subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
print('Verified source SHA:', checked_out_sha)


## 2. Inspect Kaggle and require its CUDA-enabled PyTorch

Keep Kaggle's supplied PyTorch installation; do not install the local requirements file over it. Record package versions and GPU information.


In [ ]:
from importlib.metadata import version
from dataclasses import asdict
from datetime import datetime, timezone
from time import perf_counter
from math import ceil
import hashlib, json, platform
import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib.pyplot as plt
import torch
from src.data import BROAD_CLASS_ORDER, sample_class_balanced_indices, stratified_split_indices, materialize_selected_rows
from src.continual_learning import TASK1_CLASS_IDS, TASK2_CLASS_IDS, prepare_two_task_dataset, select_balanced_replay
from src.models import ModelConfig, TabularTransformerClassifier
from src.evaluation import classification_metrics, forgetting_metrics
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required; enable a Kaggle GPU and restart.')
assert pa.array([1, 2]).to_pylist() == [1, 2]
assert pd.Series([1, 2]).sum() == 3
assert torch.tensor([1.0], device='cuda').is_cuda
environment = {'python': platform.python_version(), 'packages': {name: version(name) for name in ('numpy', 'pandas', 'pyarrow', 'torch', 'matplotlib')}, 'torch_cuda_build': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0)}
print('Environment:', environment)


## 3. Locate the same attached Parquet collection

Set `PARQUET_PATH` if Kaggle finds zero or multiple candidates. Hashing scans the full file. A different checksum stops the comparison rather than silently changing inputs.


In [ ]:
PARQUET_PATH = None  # e.g. '/kaggle/input/private-collection/cic-collection.parquet'
EXPECTED_INPUT_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
candidates = sorted(Path('/kaggle/input').rglob('cic-collection.parquet'))
print('Candidates:', [str(path) for path in candidates])
if PARQUET_PATH is None:
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one attached Parquet file, found {len(candidates)}; set PARQUET_PATH.')
    parquet_path = candidates[0]
else:
    parquet_path = Path(PARQUET_PATH)
    if not parquet_path.is_file() or not parquet_path.is_relative_to(Path('/kaggle/input')):
        raise ValueError('PARQUET_PATH must name an attached file under /kaggle/input')
start = perf_counter(); digest = hashlib.sha256()
with parquet_path.open('rb') as source_file:
    for chunk in iter(lambda: source_file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
input_sha256 = digest.hexdigest(); checksum_seconds = perf_counter() - start
print('Input bytes:', parquet_path.stat().st_size, 'SHA-256:', input_sha256)
if input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError('Attached file differs from the reviewed clean/poisoning collection.')


## 4. Sample and split once; fit only on Task 1 training

The seeded sampler selects 2,000 rows per broad class but still scans the Parquet label column. The existing stratified splitter fixes the 70/15/15 partitions. `materialize_selected_rows` preserves original row IDs. The new task builder uses the existing `encode_broad_labels` and `fit_preprocessor` helpers, fitting statistics **only on Task 1 training rows**. Task 2, validation, and test never fit them. Global IDs remain 0–7: Task 1 = Benign, DDoS, DoS, Botnet, Bruteforce (0–4); Task 2 = Infiltration, Webattack, Portscan (5–7).


In [ ]:
start = perf_counter()
sample = sample_class_balanced_indices(parquet_path, per_class_limit=2000, seed=42)
sampling_seconds = perf_counter() - start
assert sample.class_order == BROAD_CLASS_ORDER
split = stratified_split_indices(sample.labels, seed=42)
start = perf_counter()
selected = materialize_selected_rows(parquet_path, sample.row_indices)
assert np.array_equal(selected.labels, sample.labels) and np.array_equal(selected.row_indices, sample.row_indices)
tasks = prepare_two_task_dataset(selected, split)
preparation_seconds = perf_counter() - start
assert tasks.task1.class_ids == TASK1_CLASS_IDS and tasks.task2.class_ids == TASK2_CLASS_IDS
assert tasks.class_names == dict(enumerate(BROAD_CLASS_ORDER))
for task_name in ('task1', 'task2'):
    task = getattr(tasks, task_name)
    for partition_name in ('train', 'validation', 'test'):
        part = getattr(task, partition_name)
        assert part.features.dtype == np.float32 and np.isfinite(part.features).all()
        assert set(np.unique(part.labels)) == set(task.class_ids)
        print(task_name, partition_name, part.features.shape, 'global IDs:', np.unique(part.labels).tolist())
train_rows = np.concatenate((tasks.task1.train.row_indices, tasks.task2.train.row_indices))
validation_rows = tasks.seen_validation.row_indices
test_rows = tasks.combined_test.row_indices
assert not np.intersect1d(train_rows, validation_rows).size
assert not np.intersect1d(train_rows, test_rows).size
assert not np.intersect1d(validation_rows, test_rows).size
print('Class mapping:', tasks.class_names, 'source availability:', sample.available_counts)


## 5. Select a clean, balanced Task 1 memory

Select 100 unique Task 1 training rows per old class with seed 42 (500 total), excluding validation/test raw row IDs. Replay stores copied transformed features and global labels. Its `as_fit_replay()` output matches the classifier's existing `(X_replay, y_replay)` argument. The sequential arm receives no memory.


In [ ]:
replay = select_balanced_replay(tasks.task1.train, class_ids=TASK1_CLASS_IDS, per_class=100, seed=42, forbidden_row_indices=np.concatenate((validation_rows, test_rows)))
assert replay.features.shape[0] == 500 and len(np.unique(replay.row_indices)) == 500
assert not np.intersect1d(replay.row_indices, validation_rows).size
assert not np.intersect1d(replay.row_indices, test_rows).size
assert set(replay.row_indices).issubset(set(tasks.task1.train.row_indices))
print('Replay counts:', replay.class_counts, 'unique original rows:', len(replay.row_indices))


## 6. Create the auditable output folder

Store source SHA, environment, input checksum, seeds, selected/split raw row indices, Task 1-only preprocessing state, and replay membership. The dataset and generated artifacts remain outside Git.


In [ ]:
OUTPUT_DIR = Path('/kaggle/working') / ('review2_continual_replay_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
np.savez_compressed(OUTPUT_DIR / 'row_indices.npz', selected_original=sample.row_indices, train_positions=split.train, validation_positions=split.validation, test_positions=split.test, task1_train_original=tasks.task1.train.row_indices, task2_train_original=tasks.task2.train.row_indices, validation_original=validation_rows, test_original=test_rows)
np.savez_compressed(OUTPUT_DIR / 'replay_indices.npz', original_rows=replay.row_indices, task1_train_positions=replay.source_train_positions, labels=replay.labels)
np.savez_compressed(OUTPUT_DIR / 'replay_buffer.npz', features=replay.features, labels=replay.labels, original_rows=replay.row_indices)
preprocessing = {'fit_scope': 'Task 1 training rows only', 'feature_names': list(tasks.preprocessor.feature_names), 'imputation_values': tasks.preprocessor.imputation_values.tolist(), 'means': tasks.preprocessor.means.tolist(), 'scales': tasks.preprocessor.scales.tolist()}
(OUTPUT_DIR / 'preprocessing.json').write_text(json.dumps(preprocessing, indent=2), encoding='utf-8')
manifest = {'purpose': 'clean class-incremental Task 1 to Task 2: sequential versus balanced replay', 'source_sha': checked_out_sha, 'environment': environment, 'input': {'path': str(parquet_path), 'bytes': parquet_path.stat().st_size, 'sha256': input_sha256, 'provenance': 'combined flow collection; per-row source provenance unverified'}, 'sampling': {'per_class_limit': 2000, 'seed': 42, 'available_counts': sample.available_counts}, 'split': {'seed': 42, 'fractions': [0.70, 0.15, 0.15]}, 'class_names': {str(key): name for key, name in tasks.class_names.items()}, 'task1_class_ids': list(TASK1_CLASS_IDS), 'task2_class_ids': list(TASK2_CLASS_IDS), 'preprocessing_fit_scope': 'Task 1 training only', 'replay': {'seed': 42, 'per_old_class': 100, 'counts': replay.class_counts, 'total_rows': len(replay.row_indices), 'source': 'Task 1 training only'}, 'timings_seconds': {'checksum': checksum_seconds, 'sampling': sampling_seconds, 'materialization_and_preparation': preparation_seconds}, 'model_selection': 'Task 1: clean Task 1 validation; Task 2: same clean seen-class validation pool for both arms; test excluded', 'arms': {}}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Output folder:', OUTPUT_DIR)


## 7. Train a fresh five-output Task 1 model

This model starts from scratch, not the eight-class static checkpoint. It sees only Task 1 training/validation rows and must retain class IDs 0–4. The existing model restores its minimum validation-loss epoch; test is inspected only after that selection. Save the best Task 1 checkpoint as the common starting point for both Task 2 arms.


In [ ]:
model_config = ModelConfig(num_features=len(tasks.feature_names), num_classes=5, hidden_dim=64, num_heads=4, num_layers=2, mlp_dim=128, batch_size=128, learning_rate=0.001, epochs=20, early_stopping_patience=5, device='cuda', mixed_precision=False, seed=42)
manifest['model_config_task1'] = asdict(model_config)
task1_model = TabularTransformerClassifier(model_config, class_ids=TASK1_CLASS_IDS)
assert task1_model.class_ids == TASK1_CLASS_IDS
torch.cuda.synchronize(); start = perf_counter()
task1_history = task1_model.fit(tasks.task1.train.features, tasks.task1.train.labels, tasks.task1.validation.features, tasks.task1.validation.labels, verbose=True)
torch.cuda.synchronize(); task1_train_seconds = perf_counter() - start
assert task1_model.class_ids == TASK1_CLASS_IDS, 'Future classes entered Task 1'
assert task1_history.best_epoch == int(np.argmin(task1_history.validation_loss)) + 1
assert np.isfinite(task1_history.train_loss + task1_history.validation_loss + task1_history.validation_accuracy).all()
task1_checkpoint = OUTPUT_DIR / 'task1_best_validation.pt'
task1_model.save(task1_checkpoint)
task1_predictions = task1_model.predict(tasks.task1.test.features)
task1_before = classification_metrics(tasks.task1.test.labels, task1_predictions, class_ids=TASK1_CLASS_IDS, class_names=tasks.class_names, benign_class_id=0)
task1_reload = TabularTransformerClassifier.load(task1_checkpoint, device='cuda')
assert task1_reload.class_ids == TASK1_CLASS_IDS
task1_reload_identical = np.array_equal(task1_reload.predict(tasks.task1.test.features), task1_predictions)
assert task1_reload_identical
task1_steps = ceil(len(tasks.task1.train.labels) / model_config.batch_size) * task1_history.epochs_completed
(OUTPUT_DIR / 'task1_history.json').write_text(json.dumps(task1_history.to_dict(), indent=2), encoding='utf-8')
(OUTPUT_DIR / 'task1_metrics.json').write_text(json.dumps(task1_before, indent=2), encoding='utf-8')
manifest['task1'] = {'train_rows': len(tasks.task1.train.labels), 'validation_rows': len(tasks.task1.validation.labels), 'test_rows': len(tasks.task1.test.labels), 'class_ids': list(task1_model.class_ids), 'history': task1_history.to_dict(), 'optimizer_steps': task1_steps, 'train_seconds': task1_train_seconds, 'checkpoint': task1_checkpoint.name, 'reload_identical': bool(task1_reload_identical), 'test_metrics_before_task2': task1_before}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Task 1 best epoch:', task1_history.best_epoch, 'old-test accuracy:', task1_before['accuracy'], 'macro-F1:', task1_before['macro_f1'])


## 8. Reload twice and verify identical expanded initial weights

Independent checkpoint reloads start both arms from the same saved Task 1 weights. Each expands from five to eight outputs in global-ID order under the same seed; hashes verify all expanded initial tensors match. The validation pool contains clean rows from **all classes seen by Task 2** for both arms. This assumes trusted old/new validation data and is not a mitigation method.


In [ ]:
def state_hash(model):
    digest = hashlib.sha256()
    for key, value in sorted(model.network.state_dict().items()):
        digest.update(key.encode()); digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()
arm_models = {}
initial_hashes = {}
for arm_name in ('sequential', 'replay'):
    model = TabularTransformerClassifier.load(task1_checkpoint, device='cuda')
    assert model.class_ids == TASK1_CLASS_IDS
    model.add_classes(TASK2_CLASS_IDS)
    assert model.class_ids == TASK1_CLASS_IDS + TASK2_CLASS_IDS
    arm_models[arm_name] = model
    initial_hashes[arm_name] = state_hash(model)
assert initial_hashes['sequential'] == initial_hashes['replay'], 'Expanded initial states differ'
manifest['task2_expanded_initial_sha256'] = initial_hashes['sequential']
print('Matching expanded initial weights:', initial_hashes)


## 9. Train sequential and balanced-replay Task 2 arms

Both arms use the same Task 2 rows, hyperparameters, initial state, and clean seen-class validation pool. The replay arm adds 500 stored Task 1 rows through the existing `fit(..., replay=(X, y))` contract. More rows imply more batches/optimizer steps; record both rather than attributing every difference solely to replay's content. Per-epoch progress is printed. No test row selects a checkpoint.


In [ ]:
arm_histories = {}
for arm_name, model in arm_models.items():
    arm_dir = OUTPUT_DIR / arm_name; arm_dir.mkdir()
    replay_argument = None if arm_name == 'sequential' else replay.as_fit_replay()
    replay_rows = 0 if replay_argument is None else len(replay_argument[1])
    total_training_rows = len(tasks.task2.train.labels) + replay_rows
    torch.manual_seed(model_config.seed); torch.cuda.manual_seed_all(model_config.seed)
    torch.cuda.synchronize(); start = perf_counter()
    history = model.fit(tasks.task2.train.features, tasks.task2.train.labels, tasks.seen_validation.features, tasks.seen_validation.labels, replay=replay_argument, verbose=True)
    torch.cuda.synchronize(); elapsed = perf_counter() - start
    assert model.class_ids == TASK1_CLASS_IDS + TASK2_CLASS_IDS
    assert history.best_epoch == int(np.argmin(history.validation_loss)) + 1
    assert np.isfinite(history.train_loss + history.validation_loss + history.validation_accuracy).all()
    checkpoint = arm_dir / 'best_validation.pt'
    model.save(checkpoint)
    steps = ceil(total_training_rows / model.config.batch_size) * history.epochs_completed
    arm_histories[arm_name] = history
    (arm_dir / 'history.json').write_text(json.dumps(history.to_dict(), indent=2), encoding='utf-8')
    manifest['arms'][arm_name] = {'task2_new_rows': len(tasks.task2.train.labels), 'replay_rows': replay_rows, 'training_rows_per_epoch': total_training_rows, 'optimizer_steps_per_epoch': ceil(total_training_rows / model.config.batch_size), 'optimizer_steps_total': steps, 'training_seconds': elapsed, 'config': asdict(model.config), 'history': history.to_dict(), 'checkpoint': str(checkpoint.relative_to(OUTPUT_DIR)), 'expanded_initial_sha256': initial_hashes[arm_name]}
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(arm_name, 'epochs:', history.epochs_completed, 'best:', history.best_epoch, 'rows/epoch:', total_training_rows, 'optimizer steps:', steps, 'seconds:', elapsed)


## 10. Evaluate after selection and verify both reloads

Compare Task 1 test predictions before and after each arm, Task 2 test, and combined test. The old-class macro-F1 uses the same five classes and old test rows before/after; predictions into new classes remain errors. Forgetting is **before minus after**, so negative values are improvements. Benign false-positive rate is reported. Checkpoint reload must reproduce combined-test predictions. No model setting is tuned from these test results.


In [ ]:
ALL_IDS = TASK1_CLASS_IDS + TASK2_CLASS_IDS
def checked_predictions(model, part):
    probabilities = model.predict_proba(part.features)
    predictions = model.predict(part.features)
    assert probabilities.shape == (len(part.labels), len(model.class_ids))
    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5)
    assert np.array_equal(predictions, np.asarray(model.class_ids)[probabilities.argmax(axis=1)])
    return predictions
def focused_metrics(truth, predictions, focus_ids, *, benign=False):
    result = classification_metrics(truth, predictions, class_ids=ALL_IDS, class_names=tasks.class_names, benign_class_id=0 if benign else None)
    result['focused_class_ids'] = list(focus_ids)
    result['focused_macro_f1'] = float(np.mean([result['per_class'][str(class_id)]['f1'] for class_id in focus_ids]))
    return result
summary = {}
for arm_name, model in arm_models.items():
    arm_dir = OUTPUT_DIR / arm_name
    old_predictions = checked_predictions(model, tasks.task1.test)
    new_predictions = checked_predictions(model, tasks.task2.test)
    combined_predictions = checked_predictions(model, tasks.combined_test)
    old_metrics = focused_metrics(tasks.task1.test.labels, old_predictions, TASK1_CLASS_IDS, benign=True)
    new_metrics = focused_metrics(tasks.task2.test.labels, new_predictions, TASK2_CLASS_IDS)
    combined_metrics = classification_metrics(tasks.combined_test.labels, combined_predictions, class_ids=ALL_IDS, class_names=tasks.class_names, benign_class_id=0)
    forgetting = forgetting_metrics(tasks.task1.test.labels, task1_predictions, old_predictions, old_class_ids=TASK1_CLASS_IDS, all_class_ids=ALL_IDS)
    checkpoint = arm_dir / 'best_validation.pt'
    reloaded = TabularTransformerClassifier.load(checkpoint, device='cuda')
    reload_identical = reloaded.class_ids == model.class_ids and np.array_equal(reloaded.predict(tasks.combined_test.features), combined_predictions)
    assert reload_identical, f'{arm_name} checkpoint reload mismatch'
    metrics = {'task1_before': task1_before, 'task1_after': old_metrics, 'task2_after': new_metrics, 'combined_after': combined_metrics, 'forgetting': forgetting}
    (arm_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    manifest['arms'][arm_name]['metrics'] = metrics
    manifest['arms'][arm_name]['reload_identical'] = bool(reload_identical)
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    summary[arm_name] = {'old_accuracy_after': old_metrics['accuracy'], 'old_macro_f1_after': old_metrics['focused_macro_f1'], 'task2_accuracy': new_metrics['accuracy'], 'task2_macro_f1': new_metrics['focused_macro_f1'], 'combined_accuracy': combined_metrics['accuracy'], 'combined_macro_f1': combined_metrics['macro_f1'], 'accuracy_forgetting': forgetting['accuracy_forgetting'], 'macro_f1_forgetting': forgetting['macro_f1_forgetting'], 'benign_fpr': old_metrics['benign_false_positive']['rate'], 'optimizer_steps': manifest['arms'][arm_name]['optimizer_steps_total']}
    print(arm_name, 'old accuracy:', old_metrics['accuracy'], 'old macro-F1:', old_metrics['focused_macro_f1'], 'Task 2 accuracy:', new_metrics['accuracy'], 'combined accuracy:', combined_metrics['accuracy'], 'forgetting:', forgetting)
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(pd.DataFrame(summary).T)


## 11. Plot loss curves and collect the evidence

Curves show observed optimization and best clean-validation epochs. Send the executed notebook plus the timestamped output folder back for review before replay poisoning or mitigation. A single seed and balanced subset do not establish general continual-learning performance.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for axis, name, history in zip(axes, ('Task 1', 'Sequential', 'Replay'), (task1_history, arm_histories['sequential'], arm_histories['replay'])):
    epochs = np.arange(1, history.epochs_completed + 1)
    axis.plot(epochs, history.train_loss, label='Train loss')
    axis.plot(epochs, history.validation_loss, label='Clean validation loss')
    axis.axvline(history.best_epoch, color='gray', linestyle='--', label='Best epoch')
    axis.set(title=name, xlabel='Epoch', ylabel='Cross-entropy loss')
    axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160)
plt.show()
manifest['artifacts'] = ['manifest.json', 'summary.json', 'row_indices.npz', 'replay_indices.npz', 'replay_buffer.npz', 'preprocessing.json', 'task1_best_validation.pt', 'task1_history.json', 'task1_metrics.json', 'sequential/history.json', 'sequential/metrics.json', 'sequential/best_validation.pt', 'replay/history.json', 'replay/metrics.json', 'replay/best_validation.pt', 'training_curves.png']
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Source SHA:', checked_out_sha, 'input SHA-256:', input_sha256, 'GPU:', environment['gpu_name'])
print('Task 1 reload and both Task 2 reloads: identical')
print('Output folder:', OUTPUT_DIR)
